In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
example_dir = 'TCJA_TMD##frisch##0.4##zeta_D##0.4##g_y_annual##0.02##tG1##20/'
base_dir = os.path.join(CUR_DIR, example_dir, "OUTPUT_BASELINE")
reform_dir = os.path.join(CUR_DIR, example_dir, "OUTPUT_REFORM")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56,-5.56
1,IIT: Pct Change due to behavior,1.17,1.18,1.20,1.21,1.23,1.24,1.25,1.26,1.27,1.27,1.23,1.50
2,IIT: Pct Change due to macro,0.01,-0.01,-0.02,-0.03,-0.05,-0.06,-0.08,-0.09,-0.11,-0.13,-0.06,-0.07
3,IIT: Overall Pct Change in taxes,-4.45,-4.45,-4.44,-4.44,-4.45,-4.45,-4.46,-4.46,-4.47,-4.48,-4.46,-4.20
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.19,0.36,0.55,0.67,0.76,0.82,0.86,0.88,0.88,0.87,0.68,3.22
6,CIT: Pct Change due to macro,1.12,0.90,0.68,0.53,0.41,0.32,0.25,0.20,0.17,0.15,0.47,-1.82
7,CIT: Overall Pct Change in taxes,1.31,1.26,1.24,1.20,1.17,1.14,1.11,1.08,1.05,1.02,1.16,1.35
8,All: Pct Change due to tax rates,-5.26,-5.26,-5.26,-5.26,-5.26,-5.26,-5.26,-5.26,-5.26,-5.26,-5.26,-5.27
9,All: Pct Change due to behavior,1.12,1.13,1.17,1.18,1.20,1.21,1.23,1.24,1.24,1.25,1.20,1.60


In [6]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.27,-0.28,-0.30,-0.31,-0.32,-0.33,-0.35,-0.36,-0.38,-0.39,-3.30
9,Rev Change Due to Behavior,0.06,0.06,0.07,0.07,0.07,0.08,0.08,0.09,0.09,0.09,0.75
10,Rev Change Due to Macro,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.02
11,Total Revenue Change,-0.21,-0.22,-0.24,-0.25,-0.25,-0.26,-0.28,-0.29,-0.30,-0.31,-2.61


In [8]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.24,-0.26,-0.28,-0.29,-0.30,-0.31,-0.32,-0.33,-0.35,-0.36,-3.03
1,Rev Change Due to Behavior,0.05,0.05,0.06,0.06,0.07,0.07,0.07,0.08,0.08,0.08,0.67
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.03
3,Total Revenue Change,-0.19,-0.21,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-0.28,-0.29,-2.43


In [9]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)
result_df_dynamic = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_w_behresp.csv', index_col = 0)

In [11]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.23,-0.25,-0.26,-0.27,-0.29,-0.30,-0.31,-0.32,-0.34,-0.35,-2.92
1,Rev Change Due to Behavior,0.05,0.05,0.06,0.06,0.06,0.07,0.07,0.07,0.08,0.08,0.65
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.03
3,Total Revenue Change,-0.18,-0.20,-0.21,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-0.28,-2.34


In [13]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.35,-0.36,-0.36,-2.95
1,Rev Change Due to Behavior,0.05,0.05,0.05,0.06,0.06,0.06,0.07,0.07,0.07,0.08,0.61
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
3,Total Revenue Change,0.05,-0.24,-0.25,-0.25,-0.26,-0.26,-0.27,-0.28,-0.28,-0.29,-2.34


In [14]:
df_levels.to_csv('og_usa_result_w_tcja.csv')